A1c. Вложенность пунктов в кейсы: n_eff и df_design (Anglin 2026).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/statistical_inference')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
ITEMS, POS = load_items()
print()
LONG = items_long(ITEMS, POS)
print('длинная таблица:', LONG.shape)
print('пунктов на кейс (среднее): %.2f' % LONG.groupby(['arm', 'case'], observed=True).size().mean())


плеч с попунктными вердиктами: 18
общих положительных пунктов: 2249

длинная таблица: (40482, 6)
пунктов на кейс (среднее): 10.22


In [2]:
rows = []
for arm, g in LONG.groupby('arm', observed=True):
    rho, B0, k = icc_anova(g['met'].to_numpy(float), g['case'].to_numpy())
    n = len(g)
    p = float(g['met'].mean())
    deff = 1 + rho * (B0 - 1)
    rows.append({'плечо': arm, 'пунктов n': n, 'кейсов m': k, 'B0 (пунктов/кейс)': B0, 'доля met': p, 'ICC ρ': rho, 'DEFF': deff, 'n_eff': n / deff, 'n_eff/кейсов': n / deff / k})
I = pd.DataFrame(rows).set_index('плечо').sort_values('доля met')
show(I, 'Вложенность пунктов в кейсы: ICC и эффективный размер')



Вложенность пунктов в кейсы: ICC и эффективный размер
------------------------------------------------------------
                        пунктов n  кейсов m  B0 (пунктов/кейс)  доля met  ICC ρ   DEFF     n_eff  n_eff/кейсов
плечо                                                                                                         
baseline gemini              2249       220            10.2223    0.3988 0.0160 1.1472 1960.3777        8.9108
baseline deepseekv4pro       2249       220            10.2223    0.5158 0.0313 1.2886 1745.3465        7.9334
baseline qwen                2249       220            10.2223    0.5216 0.0074 1.0686 2104.5864        9.5663
baseline glm                 2249       220            10.2223    0.5411 0.0352 1.3246 1697.8869        7.7177
baseline kimi                2249       220            10.2223    0.5522 0.0368 1.3392 1679.3717        7.6335
USC@3 gpt                    2249       220            10.2223    0.5820 0.0332 1.3059 1722.1705        7.8

In [3]:
def intervals(x, n, rho, B0, m, alpha=0.05):
    p = x / n
    z = stats.norm.ppf(1 - alpha / 2)
    out = {}
    out['Wald (наив.)'] = ci_wald(p, n, z)
    out['Wilson (наив.)'] = ci_wilson(p, n, z)
    out['Agresti-Coull (наив.)'] = ci_agresti_coull(x, n, z)
    out['Clopper-Pearson (наив.)'] = ci_clopper_pearson(int(round(x)), int(n), alpha)
    deff = 1 + rho * (B0 - 1)
    var = p * (1 - p) / n * deff
    neff = p * (1 - p) / var if var > 0 else n
    out['Wald @ n_eff'] = ci_wald(p, neff, z)
    out['Agresti-Coull @ n_eff'] = ci_agresti_coull(p * neff, neff, z)
    dfd = max(m - 1, 1)
    tcrit = stats.t.ppf(1 - alpha / 2, dfd)
    neff_star = neff * (z / tcrit) ** 2
    out['Agresti-Coull @ n_eff* (+df_design)'] = ci_agresti_coull(p * neff_star, neff_star, z)
    return (p, out, neff, neff_star, dfd)
rows = []
for arm in ['P-base', 'baseline gpt', 'homog gpt', 'USC@7 gpt']:
    if arm not in ITEMS:
        continue
    g = LONG[LONG['arm'] == arm]
    rho, B0, m = icc_anova(g['met'].to_numpy(float), g['case'].to_numpy())
    p, out, neff, neff_star, dfd = intervals(g['met'].sum(), len(g), rho, B0, m)
    for k, (lo, hi) in out.items():
        rows.append({'плечо': arm, 'метод': k, 'доля': p, 'низ': lo, 'верх': hi, 'ширина': hi - lo, 'n_eff': neff, 'n_eff*': neff_star, 'df_design': dfd})
CI = pd.DataFrame(rows)
show(CI.set_index(['плечо', 'метод'])[['доля', 'низ', 'верх', 'ширина']], 'Доверительные интервалы для доли выполненных пунктов')



Доверительные интервалы для доли выполненных пунктов
------------------------------------------------------------
                                                   доля    низ   верх  ширина
плечо        метод                                                           
P-base       Wald (наив.)                        0.7528 0.7349 0.7706  0.0357
             Wilson (наив.)                      0.7528 0.7345 0.7702  0.0356
             Agresti-Coull (наив.)               0.7528 0.7345 0.7702  0.0356
             Clopper-Pearson (наив.)             0.7528 0.7344 0.7705  0.0361
             Wald @ n_eff                        0.7528 0.7347 0.7708  0.0361
             Agresti-Coull @ n_eff               0.7528 0.7343 0.7704  0.0360
             Agresti-Coull @ n_eff* (+df_design) 0.7528 0.7342 0.7705  0.0362
baseline gpt Wald (наив.)                        0.5847 0.5643 0.6051  0.0407
             Wilson (наив.)                      0.5847 0.5642 0.6049  0.0407
             Agresti-Coull 

In [4]:
piv = CI.pivot_table(index='плечо', columns='метод', values='ширина')
cmp_ = pd.DataFrame({'ширина Wald наив.': piv['Wald (наив.)'], 'ширина AC @ n_eff': piv['Agresti-Coull @ n_eff'], 'ширина AC @ n_eff*': piv['Agresti-Coull @ n_eff* (+df_design)']})
cmp_['занижение (n_eff), раз'] = cmp_['ширина AC @ n_eff'] / cmp_['ширина Wald наив.']
cmp_['занижение (n_eff*), раз'] = cmp_['ширина AC @ n_eff*'] / cmp_['ширина Wald наив.']
cmp_['вклад одного df_design'] = cmp_['ширина AC @ n_eff*'] / cmp_['ширина AC @ n_eff']
show(cmp_, 'Насколько наивный попунктный интервал слишком узок')



Насколько наивный попунктный интервал слишком узок
------------------------------------------------------------
              ширина Wald наив.  ширина AC @ n_eff  ширина AC @ n_eff*  занижение (n_eff), раз  занижение (n_eff*), раз  вклад одного df_design
плечо                                                                                                                                          
P-base                   0.0357             0.0360              0.0362                  1.0110                   1.0166                  1.0056
USC@7 gpt                0.0407             0.0466              0.0468                  1.1442                   1.1505                  1.0055
baseline gpt             0.0407             0.0417              0.0419                  1.0237                   1.0294                  1.0055
homog gpt                0.0368             0.0397              0.0399                  1.0805                   1.0865                  1.0056


In [5]:
def boot_prop(g, kind='cluster', B=2000, seed=0):
    rng = np.random.default_rng(seed)
    groups = [gg['met'].to_numpy(float) for _, gg in g.groupby('case', observed=True)]
    m = len(groups)
    out = []
    for _ in range(B):
        pick = rng.integers(0, m, m)
        if kind == 'cluster':
            vals = np.concatenate([groups[i] for i in pick])
        else:
            vals = np.concatenate([groups[i][rng.integers(0, len(groups[i]), len(groups[i]))] for i in pick])
        out.append(vals.mean())
    out = np.array(out)
    return (float(np.quantile(out, 0.025)), float(np.quantile(out, 0.975)))
rows = []
for arm in ['P-base', 'baseline gpt', 'homog gpt', 'USC@7 gpt']:
    if arm not in ITEMS:
        continue
    g = LONG[LONG['arm'] == arm]
    p = float(g['met'].mean())
    lo_s, hi_s = boot_ci(g['met'].to_numpy(float), B=2000, seed=2)
    lo_c, hi_c = boot_prop(g, 'cluster', B=2000, seed=2)
    lo_h, hi_h = boot_prop(g, 'hier', B=2000, seed=2)
    rows.append({'плечо': arm, 'доля': p, 'простой низ': lo_s, 'простой верх': hi_s, 'ш.простой': hi_s - lo_s, 'кластер низ': lo_c, 'кластер верх': hi_c, 'ш.кластер': hi_c - lo_c, 'иерарх низ': lo_h, 'иерарх верх': hi_h, 'ш.иерарх': hi_h - lo_h})
BS = pd.DataFrame(rows).set_index('плечо')
BS['кластер/простой'] = BS['ш.кластер'] / BS['ш.простой']
BS['иерарх/простой'] = BS['ш.иерарх'] / BS['ш.простой']
show(BS[['доля', 'ш.простой', 'ш.кластер', 'ш.иерарх', 'кластер/простой', 'иерарх/простой']], 'Бутстрап: простой / кластерный / иерархический')



Бутстрап: простой / кластерный / иерархический
------------------------------------------------------------
               доля  ш.простой  ш.кластер  ш.иерарх  кластер/простой  иерарх/простой
плечо                                                                               
P-base       0.7528     0.0342     0.0362    0.0491           1.0568          1.4337
baseline gpt 0.5847     0.0405     0.0423    0.0565           1.0448          1.3956
homog gpt    0.7283     0.0373     0.0411    0.0528           1.0996          1.4148
USC@7 gpt    0.5865     0.0418     0.0478    0.0584           1.1433          1.3975


In [6]:
import statsmodels.api as sm
sub = LONG[LONG['arm'].isin(['P-base', 'baseline gpt'])].copy()
sub['panel'] = (sub['arm'] == 'P-base').astype(int)
X = sm.add_constant(sub[['panel']].astype(float))
y = sub['met'].astype(float)
m_naive = sm.Logit(y, X).fit(disp=0)
m_clust = sm.Logit(y, X).fit(cov_type='cluster', cov_kwds={'groups': sub['case'].astype('category').cat.codes}, disp=0)
out = pd.DataFrame({'коэф': m_naive.params, 'SE наив.': m_naive.bse, 'p наив.': m_naive.pvalues, 'SE кластер': m_clust.bse, 'p кластер': m_clust.pvalues})
out['SE выросла, раз'] = out['SE кластер'] / out['SE наив.']
show(out, 'Логистическая модель met ~ панель: наивные и кластерные SE')



Логистическая модель met ~ панель: наивные и кластерные SE
------------------------------------------------------------
        коэф  SE наив.  p наив.  SE кластер  p кластер  SE выросла, раз
const 0.3421    0.0428   0.0000      0.0442     0.0000           1.0323
panel 0.7714    0.0650   0.0000      0.0463     0.0000           0.7133


In [7]:
save(I.reset_index(), 'effective_sample_size_and_design_effect_icc_neff.csv')
save(CI, 'effective_sample_size_and_design_effect_intervals.csv')
save(BS.reset_index(), 'effective_sample_size_and_design_effect_bootstrap.csv')


сохранено: out/effective_sample_size_and_design_effect_icc_neff.csv  (18 строк)
сохранено: out/effective_sample_size_and_design_effect_intervals.csv  (28 строк)
сохранено: out/effective_sample_size_and_design_effect_bootstrap.csv  (4 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\statistical_inference\\results\\effective_sample_size_and_design_effect_bootstrap.csv'